
# Simulator
This simulator does not reproduce exaclty the internals of the system. It is focused on demonstrating what behavior a user can expect from the protocol.

## Scope
- Lending and borrowing with interest accrual
- Soft liquidation as gradual collateral conversion
- Hard liquidation as solvency backstop
- Path-dependent behavior
- Simplified, focus on economic intuition



## Assumptions and Simplifications

- Single lender
- Single borrower
- Soft liquidation modeled as fractional collateral conversion, no arbitrage in the AMM
- Constant interest rate
- Constant LTV
- No bands in the AMM, just one liquidation range
- Simplified health factor 



# Imports

In [102]:
from dataclasses import dataclass
from pprint import pprint

# Core Components

In [ ]:
# ---------- Core Components ----------

@dataclass
class Oracle:
    price: float = 100
    def update(self, new_price: float):
        self.price = new_price


@dataclass
class Borrower:
    collateralToken: float = 5
    debt: float = 0
    borrowToken: float = 100
    soft_liquidation: bool = False
    hard_liquidation: bool = False
    
@dataclass
class Lender:
    borrowToken: float = 1000
    supplyVaultShares: float = 0

@dataclass
class LlammaLend:
    collateralToken: float = 200
    borrowToken: float = 1000000

    debt: float = 50000
    ltv: float = 0.8
    rangeWidth: float = 0.2
    liq_low: float = 0
    liq_high: float = 0
    borrowerMaxCollateral: float = 0
    borrowerCollateral: float = 0
    borrowerBorrowToken: float = 0
    interestRate: float = 0.02
    liqDiscount: float = 0.01
    ammFee: float = 0.003

    totalSupplyVaultShares: float = 50000

    def updateLtv(self, ltv: float):
        self.ltv = ltv
    
    def updateRate(self, rate: float):
        self.interestRate = rate

    def health(self, borrower: Borrower, oracle: Oracle) -> float:
        if borrower.debt == 0:
            return 1
        liquidation_value = self.borrowerCollateral * oracle.price * (1 - self.liqDiscount) + self.borrowerBorrowToken
        return liquidation_value / borrower.debt - 1


    def borrow(self, borrower: Borrower, debt: float, collateral: float, oracle: Oracle):
        effective_ltv = debt/(collateral*oracle.price)
        assert effective_ltv <= self.ltv and collateral <= borrower.collateralToken
        self.liq_high = oracle.price*effective_ltv/self.ltv
        self.liq_low = self.liq_high*(1 - self.rangeWidth)
        borrower.collateralToken -= collateral
        borrower.borrowToken += debt
        borrower.debt = debt
        self.borrowToken -= debt
        self.debt += debt
        self.borrowerCollateral = collateral
        self.borrowerMaxCollateral = collateral
        self.collateralToken += collateral

    def repay(self, borrower: Borrower, debt: float, oracle: Oracle):
        assert debt <= borrower.borrowToken and debt <= borrower.debt
        borrower.debt -= debt 
        borrower.borrowToken -= debt
        self.debt -= debt
        self.borrowToken += debt
        if borrower.debt == 0:
            borrower.collateralToken += self.borrowerCollateral
            borrower.borrowToken += self.borrowerBorrowToken
            self.collateralToken -= self.borrowerCollateral
            self.borrowerCollateral = 0
            self.borrowerMaxCollateral = 0
            self.borrowerBorrowToken = 0
        elif not borrower.soft_liquidation:
            effective_ltv = borrower.debt/(self.borrowerCollateral*oracle.price + self.borrowerBorrowToken) 
            self.liq_high = oracle.price*effective_ltv/self.ltv
            self.liq_low = self.liq_high*(1 - self.rangeWidth)


    def lend(self, lender: Lender, assets: float):
        assert assets <= lender.borrowToken
        shares =(self.totalSupplyVaultShares * assets)/(self.borrowToken + self.debt)
        lender.supplyVaultShares += shares
        lender.borrowToken -= assets
        self.totalSupplyVaultShares += shares
        self.borrowToken += assets

    def redeemSupplyVaultShares(self, lender: Lender, shares: float):
        assert shares <= lender.supplyVaultShares
        assets = shares*(self.borrowToken + self.debt)/self.totalSupplyVaultShares
        lender.supplyVaultShares -= shares
        lender.borrowToken += assets
        self.totalSupplyVaultShares -= shares
        self.borrowToken -= assets
    

@dataclass
class Simulation:

    def step(self, market: LlammaLend, borrower: Borrower, oracle: Oracle):
        # Hard liquidation
        if borrower.hard_liquidation:
            liquidation_price = oracle.price*(1 - market.liqDiscount)*(1 - market.ammFee)
            market.collateralToken -= market.borrowerCollateral
            market.borrowToken += market.borrowerCollateral*liquidation_price
            market.borrowToken += market.borrowerBorrowToken
            market.debt -= market.borrowerCollateral*liquidation_price
            market.debt -= market.borrowerBorrowToken
            market.borrowerCollateral = 0
            market.borrowerBorrowToken = 0 
            borrower.debt = 0
            return

        # Interest accrual
        borrower.debt *= (1 + market.interestRate)
        market.debt *= (1 + market.interestRate)

        # Soft liquidation
        if borrower.debt != 0 and market.liq_low <= oracle.price <= market.liq_high:
            borrower.soft_liquidation = True
            
            target_frac = 1 - (market.liq_high - oracle.price)/(market.liq_high - market.liq_low)
            current_frac = market.borrowerCollateral / market.borrowerMaxCollateral
            
            if target_frac < current_frac:
                delta = current_frac - target_frac
                collateral_to_sell = min(delta*market.borrowerMaxCollateral, market.borrowerCollateral)
                market.borrowerCollateral -= collateral_to_sell
                market.borrowerBorrowToken += collateral_to_sell*oracle.price*(1 - market.ammFee)
                market.collateralToken -= collateral_to_sell
            elif target_frac > current_frac:
                delta = target_frac - current_frac
                collateral_to_buy = min(delta*market.borrowerMaxCollateral, market.borrowerBorrowToken/(oracle.price*(1 + market.ammFee)))
                market.borrowerCollateral += collateral_to_buy
                market.borrowerBorrowToken -= collateral_to_buy*oracle.price*(1 + market.ammFee)
                market.collateralToken += collateral_to_buy

        else:
            borrower.soft_liquidation = False
        
        if market.health(borrower, oracle) <= 0:
            borrower.hard_liquidation = True
            

        



## Simulation Helper


In [104]:

def run_simulation(market, lender, borrower, simulation, oracle, price_path):
    log = []
    for price in price_path:
        oracle.update(price)
        simulation.step(market, borrower, oracle)
        log.append({
            "price": oracle.price,
            "lender": {
                "borrowToken": lender.borrowToken,
                "supplyVaultShares": lender.supplyVaultShares
            }, 
            "borrower": {
                "collateralToken": borrower.collateralToken,
                "debt": borrower.debt,
                "borrowToken": borrower.borrowToken,
                "soft_liquidation": borrower.soft_liquidation,
                "hard_liquidation": borrower.hard_liquidation,
                "health": market.health(borrower, oracle)
            },
            "market": {
                "collateralToken": market.collateralToken,
                "borrowToken": market.borrowToken,
                "debt": market.debt,
                "liq_low": market.liq_low,
                "liq_high": market.liq_high,
                "borrowerCollateral": market.borrowerCollateral,
                "borrowerBorrowToken": market.borrowerBorrowToken,
                "totalSupplyVaultShares": market.totalSupplyVaultShares
            }
        })
    return log



# Example Scenarios


### Lending

In [105]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [100, 100, 100, 100, 100]
market.lend(lender, 1000)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)

price_path = [100]
market.redeemSupplyVaultShares(lender, lender.supplyVaultShares)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {5} ---")
    pprint(step)




--- Step 0 ---
{'borrower': {'borrowToken': 100,
              'collateralToken': 5,
              'debt': 0.0,
              'hard_liquidation': False,
              'health': 1,
              'soft_liquidation': False},
 'lender': {'borrowToken': 0, 'supplyVaultShares': 47.61904761904762},
 'market': {'borrowToken': 1001000,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 0,
            'collateralToken': 200,
            'debt': 51000.0,
            'liq_high': 0,
            'liq_low': 0,
            'totalSupplyVaultShares': 50047.619047619046},
 'price': 100}

--- Step 1 ---
{'borrower': {'borrowToken': 100,
              'collateralToken': 5,
              'debt': 0.0,
              'hard_liquidation': False,
              'health': 1,
              'soft_liquidation': False},
 'lender': {'borrowToken': 0, 'supplyVaultShares': 47.61904761904762},
 'market': {'borrowToken': 1001000,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 0

### Borrowing

Prices decrease, enter soft liquidation, then hard liquidation

In [106]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [60, 50, 47.5, 45, 42.5, 40, 40]
oracle.update(60)
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)



--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 0.4558823529411764,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 60}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 0.18944636678200677,
              'soft_liquidation': True},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'

Prices decrease, enter soft liquidation, then fully repay loan, avoiding hard liquidation

In [107]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [60, 50, 47.5, 45]
oracle.update(60)
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)

price_path = [42.5, 40]
market.repay(borrower, borrower.debt, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in zip([4, 5], logs):
    print(f"\n--- Step {i} ---")
    pprint(step)


--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 0.4558823529411764,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 60}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 0.18944636678200677,
              'soft_liquidation': True},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'

Prices decrease, enter soft liquidation, partially repay to improve health and delay hard liquidation while retaining some exposure to the collateral 

In [108]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [60, 50, 47.5, 45]
oracle.update(60)
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)

price_path = [42.5, 40]
market.repay(borrower, 200, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in zip([4, 5], logs):
    print(f"\n--- Step {i} ---")
    pprint(step)


--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 0.4558823529411764,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 60}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 0.18944636678200677,
              'soft_liquidation': True},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'

Prices decrease, preventively repay partially, health improves, bands shift so soft liquidation is delayed or even avoided

In [109]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [100, 90, 80]
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)

price_path = [70, 60]
market.repay(borrower, 180, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in zip([3, 4], logs):
    print(f"\n--- Step {i} ---")
    pprint(step)


--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 1.426470588235294,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 100}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 1.1410034602076125,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'

Price decrease, enter soft liquidation, then price recover. Position recovered despite some losses, user kept exposure to collateral

In [110]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [60, 50, 47.5, 45, 46, 48, 50]
oracle.update(60)
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)



--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 0.4558823529411764,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 60}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 0.18944636678200677,
              'soft_liquidation': True},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'

Price decrease and recovers in the same range, but we see the outcome is slightly different

In [111]:
oracle = Oracle()
sim = Simulation()
market = LlammaLend()
lender = Lender()
borrower = Borrower()

price_path = [60, 50, 44, 45, 49, 50]
oracle.update(60)
market.borrow(borrower, 200, 5, oracle)
logs = run_simulation(market, lender, borrower, sim, oracle, price_path)
for i, step in enumerate(logs):
    print(f"\n--- Step {i} ---")
    pprint(step)


--- Step 0 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 204.0,
              'hard_liquidation': False,
              'health': 0.4558823529411764,
              'soft_liquidation': False},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral': 5,
            'collateralToken': 205,
            'debt': 51204.0,
            'liq_high': 50.0,
            'liq_low': 40.0,
            'totalSupplyVaultShares': 50000},
 'price': 60}

--- Step 1 ---
{'borrower': {'borrowToken': 300,
              'collateralToken': 0,
              'debt': 208.08,
              'hard_liquidation': False,
              'health': 0.18944636678200677,
              'soft_liquidation': True},
 'lender': {'borrowToken': 1000, 'supplyVaultShares': 0},
 'market': {'borrowToken': 999800,
            'borrowerBorrowToken': 0,
            'borrowerCollateral'